In [1]:
### 전처리를 하기위한 import
import pandas as pd
import numpy as np

In [9]:
import os

# 1. 파일 경로와 해당 파일의 연도를 딕셔너리로 깔끔하게 정의
file_paths = {
    '2023': "../data/raw/마포구_상권_분석/mapo_store_gis_202306.csv",
    '2024': "../data/raw/마포구_상권_분석/mapo_store_gis_202406.csv",
    '2025': "../data/raw/마포구_상권_분석/mapo_store_gis_202506.csv"
}

dfs_to_concat = []

# 2. 반복문을 돌며 파일 로드 + 연도 컬럼 추가 + 인덱스 정리를 한 번에 처리
for year, path in file_paths.items():
    try:
        # 한글 깨짐 방지를 위해 우선 utf-8로 로드 시도
        df = pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        # 인코딩 에러 발생 시 cp949로 재시도
        df = pd.read_csv(path, encoding='cp949')
        
    # 🌟 복사본을 만들고 'year' 컬럼을 맨 왼쪽에 추가
    df_copy = df.copy()
    df_copy['year'] = year
    
    # 혹시 '행정동명'이 인덱스로 묶여있다면 컬럼으로 해제
    if df_copy.index.name == '행정동명':
        df_copy = df_copy.reset_index()
    elif '행정동명' not in df_copy.columns and df_copy.index.name is None:
        # 인덱스 이름이 없지만 행정동명이 인덱스에 들어가 있는 경우 대비
        df_copy = df_copy.reset_index().rename(columns={'index': '행정동명'})
        
    dfs_to_concat.append(df_copy)
    print(f"✅ {year}년 데이터 로드 완료 (Shape: {df_copy.shape})")

# 3. 판다스 concat으로 세로로 층층이 쌓아서 결합
df_master_3years = pd.concat(dfs_to_concat, axis=0, ignore_index=True)

# 4. 'year'와 '행정동명' 컬럼이 눈에 잘 띄도록 맨 앞으로 정렬
essential_cols = ['year', '행정동명']
other_cols = [col for col in df_master_3years.columns if col not in essential_cols]
df_master_3years = df_master_3years[essential_cols + other_cols]

# 5. 최종 통합된 하나의 마스터 파일로 현재 폴더에 저장
output_file = "mapo_store_gis_2023_2025.csv"
df_master_3years.to_csv(output_file, index=False, encoding='utf-8-sig')

print("=" * 60)
print(f"🎉 3개년 데이터 통합 마스터 파일 생성 완료!")
print(f"💾 저장 경로: {output_file}")
print(f"📊 최종 데이터 규모: {df_master_3years.shape} (행, 열)")
print("=" * 60)

# 데이터가 위아래로 잘 합쳐졌는지 상단/하단 확인
display(df_master_3years.head(3))
display(df_master_3years.tail(3))

✅ 2023년 데이터 로드 완료 (Shape: (27402, 8))
✅ 2024년 데이터 로드 완료 (Shape: (26914, 8))
✅ 2025년 데이터 로드 완료 (Shape: (30468, 8))
🎉 3개년 데이터 통합 마스터 파일 생성 완료!
💾 저장 경로: mapo_store_gis_2023_2025.csv
📊 최종 데이터 규모: (84784, 8) (행, 열)


,year,행정동명,시군구명,상권업종중분류명,경도,위도,건물명,상호명
0,2023,성산1동,마포구,기타 교육,126.908665,37.562237,청운빌라,볼벤
1,2023,망원1동,마포구,한식,126.909004,37.556086,정보없음,로스트피그
2,2023,서교동,마포구,섬유·의복·신발 소매,126.926114,37.554693,정보없음,레몬


,year,행정동명,시군구명,상권업종중분류명,경도,위도,건물명,상호명
84781,2025,서교동,마포구,기타 교육,126.923547,37.559674,정보없음,살사엠
84782,2025,서교동,마포구,기타 간이,126.922695,37.552268,정보없음,틈새라면홍대놀이터점
84783,2025,서교동,마포구,장식품 소매,126.913959,37.551496,메세나폴리스,인사갤러리
